<a href="https://colab.research.google.com/github/ardhyan15/BIG-DATA-PRAKTIKUM-5-7/blob/main/project_kelompok6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
# ========== INSTALL & IMPORT (Jalankan sekali) ==========
!pip install ipywidgets -q

import ipywidgets as widgets
from IPython.display import display, clear_output
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.preprocessing import LabelEncoder
import io

# ========== BUAT TAMPILAN INTERAKTIF (TANPA TERMINAL) ==========
print("📥 Silakan upload CSV atau biarkan kosong (pakai data default)")

# Widget upload file
upload = widgets.FileUpload(
    accept='.csv',
    multiple=False,
    description='Upload CSV'
)

# Widget pilih algoritma
algo_dropdown = widgets.Dropdown(
    options=['Decision Tree', 'K-NN'],
    value='Decision Tree',
    description='Algoritma:'
)

# Widget tombol
btn = widgets.Button(
    description='🚀 Jalankan Training',
    button_style='success'
)

# Tempat output
output = widgets.Output()

def proses_data(b):
    with output:
        clear_output(wait=True)
        print("⏳ Sedang memproses...")

        # 1. Ambil data (upload atau default)
        if upload.value:
            # Jika upload file
            uploaded_file = list(upload.value.values())[0]
            content = uploaded_file['content']
            df = pd.read_csv(io.BytesIO(content))
            print("✅ Dataset berhasil diupload!")
        else:
            # Data default Pima Indian Diabetes
            url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv"
            columns = ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness',
                       'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age', 'Outcome']
            df = pd.read_csv(url, header=None, names=columns)
            print("ℹ️ Menggunakan dataset DEFAULT (Pima Indians Diabetes)")

        print(f"📊 Total data: {df.shape[0]} baris, {df.shape[1]} kolom")
        display(df.head())

        # 2. Preprocessing (isi nilai kosong)
        cols_to_check = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
        for col in cols_to_check:
            if col in df.columns:
                df[col] = df[col].replace(0, np.nan)
                df[col] = df[col].fillna(df[col].median())

        # Encode kolom teks
        le = LabelEncoder()
        for col in df.select_dtypes(include=['object']).columns:
            df[col] = le.fit_transform(df[col])

        # 3. Tentukan target (asumsi kolom terakhir adalah target)
        target_col = df.columns[-1]
        print(f"🎯 Target yang diprediksi: {target_col}")

        X = df.drop(columns=[target_col])
        y = df[target_col]

        if y.nunique() > 10:
            print("⚠️ Target adalah angka kontinu, ubah ke kategorikal!")
            return

        # 4. Split & Training
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

        if algo_dropdown.value == "Decision Tree":
            model = DecisionTreeClassifier(max_depth=5, random_state=42)
        else:
            model = KNeighborsClassifier(n_neighbors=7)

        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        akurasi = accuracy_score(y_test, y_pred)

        print(f"\n✅ AKURASI MODEL: {akurasi:.2%}\n")

        # 5. Confusion Matrix
        fig, ax = plt.subplots(figsize=(6,4))
        cm = confusion_matrix(y_test, y_pred)
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax)
        ax.set_title('Confusion Matrix')
        ax.set_xlabel('Prediksi')
        ax.set_ylabel('Aktual')
        plt.show()

        # 6. Classification Report
        print("\n📋 CLASSIFICATION REPORT:")
        report = classification_report(y_test, y_pred, output_dict=True)
        display(pd.DataFrame(report).transpose())

        # 7. Feature Importance (khusus Decision Tree)
        if algo_dropdown.value == "Decision Tree":
            print("\n📊 FEATURE IMPORTANCE:")
            importance = pd.DataFrame({
                'Fitur': X.columns,
                'Importance': model.feature_importances_
            }).sort_values('Importance', ascending=False)

            fig2, ax2 = plt.subplots(figsize=(8,4))
            sns.barplot(data=importance, x='Importance', y='Fitur', palette='viridis', ax=ax2)
            ax2.set_title('Pengaruh Fitur terhadap Target')
            plt.show()
        else:
            print("ℹ️ Feature Importance hanya untuk Decision Tree.")

# Hubungkan tombol ke fungsi
btn.on_click(proses_data)

# Tampilkan semua widget
display(widgets.VBox([upload, algo_dropdown, btn, output]))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 48.9 MB/s eta 0:00:00
📥 Silakan upload CSV atau biarkan kosong (pakai data default)
